# Statistics & Probability

The math you actually need for ML, in plain language.

Five sections:
1. **Descriptive statistics** — mean, median, variance, correlation
2. **Probability basics** — single events, conditional probability, Bayes
3. **Probability distributions** — normal, Bernoulli, binomial
4. **Key concepts for ML** — bias vs variance, p-values, overfitting, cross-validation
5. **Metrics you'll use daily** — MSE, MAE, accuracy, precision, recall, F1

In [1]:
import numpy as np
import pandas as pd

## 1. Descriptive Statistics

Summary numbers that compress a whole dataset into a few values.

### Mean (the average)

Add all the values, divide by how many there are.

$$\mu = \frac{\sum x_i}{n}$$

The mean is the "balance point" of your data. Shows up everywhere — feature scaling, MSE loss, normalization.

In [2]:
np.mean([10, 20, 30, 40, 50])

30.0

### Median

The middle value once you sort the data.

Why care? The median ignores extreme values. The mean does not — a single huge value can drag the mean way up. Use the median when your data has outliers.

In [3]:
data = [1, 2, 3, 100]

print("Mean:  ", np.mean(data))    # 26.5 — one outlier pulled it way up
print("Median:", np.median(data))  # 2.5  — barely affected

Mean:   26.5
Median: 2.5


### Variance and standard deviation

How spread out are your values?

**Variance** = average squared distance from the mean.

$$\sigma^2 = \frac{\sum (x_i - \mu)^2}{n}$$

Squaring does two jobs: turns negative distances into positive ones (so they don't cancel), and punishes big gaps more than small ones.

The catch: variance is in *squared* units. **Standard deviation** = square root of variance, putting you back in normal units.

In [4]:
values = [2, 4, 6, 8]

print("Variance:           ", np.var(values))
print("Standard deviation: ", np.std(values))

Variance:            5.0
Standard deviation:  2.23606797749979


**Why this matters in ML:** something called **Z-score normalization** subtracts the mean and divides by the standard deviation. It puts every feature on the same scale, which most ML models prefer. The formula is just `(value - mean) / std`.

In [5]:
values = np.array([10, 20, 30, 40, 50])

z_scores = (values - values.mean()) / values.std()

print("Original:", values)
print("Z-scores:", z_scores.round(2))

Original: [10 20 30 40 50]
Z-scores: [-1.41 -0.71  0.    0.71  1.41]


After scaling, the mean is 0 and the standard deviation is 1. Every value is now expressed as "how many standard deviations from the mean."

### Covariance and correlation

**Covariance** tells you if two variables move together. Positive: when one goes up, the other tends to go up. Negative: they move opposite. Around zero: no relationship.

The problem with raw covariance: it depends on the units, so the number itself is hard to read.

**Correlation** = covariance rescaled to a fixed range from -1 to +1:

$$r = \frac{\text{cov}(X, Y)}{\sigma_X \, \sigma_Y}$$

How to read it:

| Value | Meaning |
|---|---|
| `r = +1` | Perfect positive — every step in X matches a step in Y |
| `r ≈ +0.7` | Strong positive |
| `r ≈ 0` | No linear relationship |
| `r ≈ -0.7` | Strong negative |
| `r = -1` | Perfect negative — when one goes up, the other goes down |

In EDA, you'll often print a correlation matrix to spot which features carry similar information.

In [6]:
df = pd.DataFrame({
    "study_hours": [1, 2, 3, 4, 5, 6, 7, 8],
    "exam_score":  [50, 55, 65, 70, 78, 85, 90, 95],
    "phone_hours": [9, 8, 7, 6, 5, 4, 3, 2],
})

df.corr().round(2)

,study_hours,exam_score,phone_hours
study_hours,1.0,1.0,-1.0
exam_score,1.0,1.0,-1.0
phone_hours,-1.0,-1.0,1.0


Reading the output: `study_hours` and `exam_score` are strongly positively correlated (r ≈ +0.99 — more study, higher score). `phone_hours` is strongly negatively correlated with both (more phone time, lower score).

## 2. Probability Basics

### What probability is

A number between 0 and 1 saying how likely something is:

$$P(A) = \frac{\text{favorable outcomes}}{\text{total outcomes}}$$

- `P(A) = 0` — impossible
- `P(A) = 1` — certain
- `P(A) + P(not A) = 1` — the **complement rule**

Rolling a fair 6-sided die: `P(rolling a 4) = 1/6 ≈ 0.167`.

In [7]:
favorable = 1   # only one face has a "4"
total = 6       # six faces total

p_four = favorable / total
print("P(rolling a 4):", round(p_four, 3))

P(rolling a 4): 0.167


### Conditional probability

The probability of A *given that* B already happened. Written `P(A | B)` and read out loud as "P of A given B."

$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)}$$

Example: of all people, what fraction have the flu? Maybe 5%. But of all people who have a fever, what fraction have the flu? Much higher — maybe 40%. Knowing "they have a fever" updates the probability.

This shows up everywhere — recommendation systems, spam filters, medical diagnosis.

### Bayes' theorem

The rule for updating your belief when new evidence comes in. The single most important equation in probabilistic ML:

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Each piece has a name:

| Term | Name | Plain English |
|---|---|---|
| `P(A)` | **Prior** | What you believed before the evidence |
| `P(B \| A)` | **Likelihood** | How likely the evidence is if your guess is right |
| `P(B)` | **Evidence** | How likely the evidence is overall |
| `P(A \| B)` | **Posterior** | Updated belief after the evidence |

**Classic example — medical testing.** A disease affects 1% of people. The test catches 99% of real cases (correctly says "positive" 99 times out of 100 when the disease is present), but also gives 5% false positives (wrongly says "positive" for 5 out of 100 healthy people). You test positive. What's the chance you actually have it?

In [8]:
# Plug numbers into Bayes' theorem
P_disease        = 0.01   # Prior: 1% of people have it
P_pos_if_sick    = 0.99   # Likelihood: catches 99% of cases
P_pos_if_healthy = 0.05   # False positive rate

# P(positive overall)
P_positive = P_pos_if_sick * P_disease + P_pos_if_healthy * (1 - P_disease)

# Bayes' rule
P_disease_given_positive = (P_pos_if_sick * P_disease) / P_positive

print("P(disease | positive test):", round(P_disease_given_positive, 3))

P(disease | positive test): 0.167


That ~17% surprises most people — they expect 99% because the test is "99% accurate." The reason it's lower: only 1% of people have the disease, so most positive results come from false alarms in the much larger healthy group.

**Why this matters for ML:** the Naive Bayes classifier applies this formula directly to predict things like whether an email is spam. You don't have to memorize the math — just remember "prior + evidence → posterior."

## 3. Probability Distributions

A **distribution** describes how likely each possible value is.

### Normal (Gaussian) distribution

The famous bell curve. Symmetric around the mean, most values near the center, fewer far out.

Two parameters define it:
- $\mu$ — the mean (where the bell is centered)
- $\sigma$ — the standard deviation (how wide the bell is)

The **empirical rule** — what fraction of values lie within how many standard deviations of the mean:

| Range | Fraction of data |
|---|---|
| Within $\pm 1\sigma$ | ~68% |
| Within $\pm 2\sigma$ | ~95% |
| Within $\pm 3\sigma$ | ~99.7% |

Heights, measurement errors, exam scores — lots of real-world quantities approximately follow this shape.

In [9]:
# Generate 1000 numbers from a normal distribution (mean=0, std=1)
samples = np.random.normal(0, 1, 1000)

print("Mean of samples:", round(samples.mean(), 2))
print("Std of samples: ", round(samples.std(), 2))

Mean of samples: -0.0
Std of samples:  1.0


### Bernoulli distribution

The simplest distribution. One trial, two outcomes — success (1) or failure (0). Probability of success is `p`.

A coin flip: `p = 0.5`. A single email being clicked: `p` = the click-through rate.

**In ML:** the output of any binary classifier (yes/no, spam/not, fraud/not) is a Bernoulli outcome.

In [10]:
# 10 flips of a biased coin (30% chance of heads)
flips = np.random.choice([0, 1], size=10, p=[0.7, 0.3])
print("Flips:", flips)

Flips: [1 0 0 0 0 0 1 0 1 0]


### Binomial distribution

Run `n` Bernoulli trials. How many successes do you get?

Flip a fair coin 10 times — you could get anywhere from 0 to 10 heads. The binomial tells you how likely each count is.

**In ML:** evaluating models. "I ran 100 predictions and got 73 right — could that easily happen by chance?" The binomial answers that.

In [11]:
# Flip 10 coins, count the heads. Repeat that experiment 5 times.
heads_per_experiment = np.random.binomial(n=10, p=0.5, size=5)
print("Heads per experiment:", heads_per_experiment)

Heads per experiment: [4 3 6 6 8]


## 4. Key Statistical Concepts for ML

### Bias vs variance — the core tradeoff

Every ML model sits somewhere on a spectrum:

- **High bias** → model is too simple to capture the pattern. *Underfits.*
- **High variance** → model is too flexible and memorizes the training data, including the noise. *Overfits.*

The goal is the sweet spot in the middle.

|  | High bias (underfit) | High variance (overfit) |
|---|---|---|
| Training error | High | Low |
| Test error | High | Very high |
| Symptom | Model can't even learn the training data | Huge gap between train and test |
| Fix | Use a more complex model, add features | Get more data, regularize, simplify the model |

A linear regression on curvy data: high bias. A 15-degree polynomial fit to 20 noisy points: high variance.

### P-value and hypothesis testing

A **p-value** is the probability of seeing your result (or something more extreme) if there was actually no real effect. Small p-value → your result is unlikely to be pure luck.

| p-value | What it suggests |
|---|---|
| `p < 0.01` | Strong evidence; very unlikely to be chance |
| `p < 0.05` | Statistically significant (the usual threshold) |
| `p ≥ 0.05` | Could just be noise — stay skeptical |

The 0.05 threshold is convention, not a law of nature.

**In ML:** comparing two models. Model A is 84% accurate; model B is 86%. Real improvement or just luck? A statistical test gives you the p-value to decide.

### Overfitting and underfitting

| | Underfitting | Overfitting |
|---|---|---|
| Symptom | Bad on training, bad on test | Great on training, bad on test |
| Cause | Model too simple | Model too complex for the data |
| How to spot it | High train error | Big gap between train and test |
| Fix | Add features, more complex model | More data, regularization, simpler model |

The most useful habit: always look at **both** train error and test error. If both are high → underfitting. If train is low but test is high → overfitting.

### Cross-validation

One train/test split gives you one number, and that number depends on which rows happened to land in the test set. **Cross-validation** averages out that randomness.

How k-fold cross-validation works:

1. Split the data into k equal folds (typically 5 or 10).
2. Train on k − 1 folds, test on the remaining one.
3. Repeat k times, holding out a different fold each time.
4. Average the k scores.

The average is a more reliable estimate of how the model will do on new data than any single split.

When you get to scikit-learn later, the whole loop becomes one line: `cross_val_score(model, X, y, cv=5)`. For now, just know the idea.

## 5. Metrics You'll Use Daily

How do you decide if a model is any good? With a metric. Different problems need different metrics.

### Regression metrics (predicting a number)

| Metric | Formula | Use when |
|---|---|---|
| **MSE** (Mean Squared Error) | $\text{mean}((y - \hat{y})^2)$ | You want to punish big errors heavily |
| **RMSE** (Root MSE) | $\sqrt{\text{MSE}}$ | You want MSE but in the same units as the data |
| **MAE** (Mean Absolute Error) | $\text{mean}(\lvert y - \hat{y} \rvert)$ | Your data has outliers; treat all errors equally |

MSE squares the errors — one bad prediction matters a lot. MAE treats all errors the same. RMSE is MSE in plain units — easier to talk about ("we're off by about $50 on average").

In [12]:
y_true = np.array([100, 200, 300, 400, 500])
y_pred = np.array([110, 195, 280, 420, 480])

errors = y_true - y_pred

mse  = np.mean(errors ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(errors))

print("MSE :", mse)
print("RMSE:", round(rmse, 2))
print("MAE :", mae)

MSE : 265.0
RMSE: 16.28
MAE : 15.0


### Classification metrics (predicting a category)

Before the metrics, the four basic outcomes when a model predicts yes/no:

| | Predicted YES | Predicted NO |
|---|---|---|
| **Actually YES** | TP (true positive) | FN (false negative — missed it) |
| **Actually NO** | FP (false positive — false alarm) | TN (true negative) |

From these four counts, you get the metrics:

| Metric | Formula | Use when |
|---|---|---|
| **Accuracy** | $(TP + TN) / \text{total}$ | Classes are balanced; all errors equally bad |
| **Precision** | $TP / (TP + FP)$ | False positives are costly (flagging a real email as spam) |
| **Recall** | $TP / (TP + FN)$ | False negatives are costly (missing a real cancer case) |
| **F1 Score** | $2 \cdot (P \cdot R) / (P + R)$ | Imbalanced classes; need a single number |

**Precision vs recall is a tradeoff.** You usually can't max both at the same time:

- A model that says "yes" for everything → perfect recall, terrible precision.
- A model that says "yes" only when 100% sure → perfect precision, terrible recall.

F1 is the harmonic mean — it forces the model to be decent at both.

In [13]:
# Cancer screening on 100 patients, 10 actually have cancer
TP = 9     # 9 real cancers correctly predicted
FP = 3     # 3 false alarms
TN = 87    # 87 healthy patients correctly cleared
FN = 1     # 1 real cancer missed

total = TP + FP + TN + FN

accuracy  = (TP + TN) / total
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * (precision * recall) / (precision + recall)

print("Accuracy :", round(accuracy, 2))
print("Precision:", round(precision, 2))
print("Recall   :", round(recall, 2))
print("F1 Score :", round(f1, 2))

Accuracy : 0.96
Precision: 0.75
Recall   : 0.9
F1 Score : 0.82


96% accuracy sounds great. But for cancer screening, recall (catching real cases) matters way more than precision (avoiding false alarms). The "right" metric depends on what mistakes cost.

### A short guide for picking the right metric

- **Regression** → start with **RMSE**. Switch to **MAE** if outliers throw it off.
- **Balanced classification** → **accuracy** is fine.
- **Imbalanced classification** (fraud, rare disease) → accuracy is misleading. Use **precision**, **recall**, or **F1**.
- **Costs aren't symmetric** → pick **precision** if false positives hurt more, **recall** if false negatives hurt more.
